# Problem 26: The Gate Automation & Damage Detection Problem

## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Learning Objectives

After completing this tier, you will be able to:
- Formulate gate automation as a mixed-integer programming problem
- Model sensor monitoring and failure detection mathematically
- Understand constraint satisfaction for maintenance scheduling
- Apply optimization to minimize downtime costs

### Problem Context

The modern logistics landscape increasingly relies on automated access control systems to manage the flow of vehicles and personnel through critical infrastructure points. Gate automation systems serve as the first line of defense and efficiency optimization for terminals, distribution centers, warehouses, and industrial facilities.

**Key Challenge**: Optimize sensor monitoring, maintenance scheduling, and failure prediction while minimizing total costs including monitoring costs and downtime costs.

In [ ]:
# Import required libraries for mathematical formulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print('🔧 Gate Automation Mathematical Formulation')
print('=' * 50)

### Mathematical Model Formulation

We formulate the gate automation and damage detection problem as a mixed-integer programming model that simultaneously optimizes sensor monitoring, maintenance scheduling, and failure prediction.

**Sets and Indices:**
\begin{align}
G &= \{1, 2, \ldots, |G|\} \text{ set of gates} \\
S &= \{1, 2, \ldots, |S|\} \text{ set of sensor types} \\
T &= \{1, 2, \ldots, |T|\} \text{ set of time periods} \\
F &= \{1, 2, \ldots, |F|\} \text{ set of failure modes}
\end{align}

In [ ]:
# Data structures for the mathematical model
@dataclass
class Gate:
    """Represents a gate in the automation system"""
    id: int
    location: str
    vehicles_per_hour: int
    criticality: float  # 0-1, higher = more important
    downtime_cost_per_hour: float

@dataclass
class Sensor:
    """Represents a sensor type"""
    id: int
    gate_id: int
    sensor_type: str  # photo_eye, ground_loop, safety_beam
    monitoring_cost: float
    failure_rate: float
    criticality: float
    maintenance_interval: int

# Create example data from the source text
gates = [
    Gate(id=1, location='Main_Entrance', vehicles_per_hour=150, criticality=0.95, downtime_cost_per_hour=15000),
    Gate(id=2, location='Exit_Gate', vehicles_per_hour=100, criticality=0.85, downtime_cost_per_hour=10000),
]

sensors = [
    Sensor(id=1, gate_id=1, sensor_type='photo_eye', monitoring_cost=5, failure_rate=0.02, criticality=0.9, maintenance_interval=8),
    Sensor(id=2, gate_id=1, sensor_type='ground_loop', monitoring_cost=9, failure_rate=0.03, criticality=0.7, maintenance_interval=12),
    Sensor(id=3, gate_id=1, sensor_type='safety_beam', monitoring_cost=7, failure_rate=0.015, criticality=1.0, maintenance_interval=6),
    Sensor(id=4, gate_id=2, sensor_type='photo_eye', monitoring_cost=5, failure_rate=0.025, criticality=0.8, maintenance_interval=8),
    Sensor(id=5, gate_id=2, sensor_type='ground_loop', monitoring_cost=9, failure_rate=0.035, criticality=0.6, maintenance_interval=12),
]

print(f'🏢 Gates: {len(gates)}')
print(f'📡 Sensors: {len(sensors)}')
print(f'⏱️  Time periods: 24 hours')
print(f'\n💰 Downtime costs:')
for gate in gates:
    print(f'  Gate {gate.id}: ${gate.downtime_cost_per_hour:,}/hour')

### Decision Variables

**Decision Variables:**
\begin{align}
x_{gst} &= \begin{cases} 1 & \text{if sensor } s \text{ at gate } g \text{ is monitored in period } t \\ 0 & \text{otherwise} \end{cases} \\
y_{gtf} &= \begin{cases} 1 & \text{if failure mode } f \text{ occurs at gate } g \text{ in period } t \\ 0 & \text{otherwise} \end{cases} \\
z_{gt} &= \begin{cases} 1 & \text{if gate } g \text{ is operational in period } t \\ 0 & \text{otherwise} \end{cases} \\
w_{gst} &= \text{sensor health index for sensor } s \text{ at gate } g \text{ in period } t
\end{align}

In [ ]:
# Mathematical model parameters
class MathematicalModel:
    """Mathematical formulation for gate automation optimization"""
    
    def __init__(self, gates: List[Gate], sensors: List[Sensor], time_periods: int = 24):
        self.gates = gates
        self.sensors = sensors
        self.time_periods = time_periods
        
        # Model parameters from source
        self.monitoring_costs = {s.id: s.monitoring_cost for s in sensors}
        self.failure_rates = {s.id: s.failure_rate for s in sensors}
        self.downtime_costs = {g.id: g.downtime_cost_per_hour for g in gates}
        self.maintenance_intervals = {s.id: s.maintenance_interval for s in sensors}
        
        # Demand patterns (vehicles per hour)
        self.demand_patterns = self._generate_demand_patterns()
        
        # Initialize solution variables
        self.monitoring_decisions = {}
        self.failure_occurrences = {}
        self.gate_operational = {}
        self.sensor_health = {}
        
    def _generate_demand_patterns(self) -> Dict[int, List[float]]:
        """Generate realistic demand patterns over 24 hours"""
        patterns = {}
        for gate in self.gates:
            pattern = []
            base_demand = gate.vehicles_per_hour
            
            for hour in range(24):
                # Peak factors: 8-10 AM and 5-7 PM
                if 8 <= hour <= 10 or 17 <= hour <= 19:
                    peak_factor = 1.8
                elif 6 <= hour <= 11 or 16 <= hour <= 21:
                    peak_factor = 1.3
                else:
                    peak_factor = 0.7
                
                pattern.append(base_demand * peak_factor)
            
            patterns[gate.id] = pattern
        
        return patterns
    
    def calculate_monitoring_cost(self) -> float:
        """Calculate total monitoring cost"""
        total_cost = 0.0
        
        for sensor in self.sensors:
            for hour in range(self.time_periods):
                if self.monitoring_decisions.get((sensor.gate_id, sensor.id, hour), 0) == 1:
                    total_cost += sensor.monitoring_cost
        
        return total_cost
    
    def calculate_downtime_cost(self) -> float:
        """Calculate total downtime cost"""
        total_cost = 0.0
        
        for gate in self.gates:
            for hour in range(self.time_periods):
                if self.gate_operational.get((gate.id, hour), 1) == 0:
                    demand = self.demand_patterns[gate.id][hour]
                    total_cost += gate.downtime_cost_per_hour * (demand / 100.0)  # Normalize by 100 vehicles
        
        return total_cost
    
    def calculate_total_cost(self) -> float:
        """Calculate total objective function value"""
        return self.calculate_monitoring_cost() + self.calculate_downtime_cost()

# Create the mathematical model
model = MathematicalModel(gates, sensors)
print(f'📐 Mathematical model created:')
print(f'  Decision variables: {len(gates) * len(sensors) * 24} monitoring decisions')
print(f'  Constraints: Sensor monitoring, failure detection, gate operations')
print(f'  Objective: Minimize total cost = monitoring_cost + downtime_cost')

### Constraints

**Key Constraints:**

1. **Sensor monitoring requirement:**
\begin{equation}
\sum_{s \in S} x_{gst} \geq 1 \quad \forall g \in G, t \in T
\end{equation}

2. **Failure detection logic:**
\begin{equation}
y_{gtf} \leq \sum_{s \in S} \lambda_{gsf} x_{gs,t-\tau_{gsf}} \quad \forall g \in G, t \in T, f \in F
\end{equation}

3. **Gate operational status:**
\begin{equation}
z_{gt} \leq 1 - \sum_{f \in F} y_{gtf} \quad \forall g \in G, t \in T
\end{equation}

4. **Sensor health bounds:**
\begin{equation}
0 \leq w_{gst} \leq 1 \quad \forall g \in G, s \in S, t \in T
\end{equation}

In [ ]:
# Implement a simplified optimization approach (enumeration for small instance)
def optimize_monitoring_schedule(model: MathematicalModel) -> Dict:
    """Find optimal monitoring schedule using simplified approach"""
    
    best_solution = None
    best_cost = float('inf')
    
    # For demonstration, use a greedy approach based on sensor criticality
    # In practice, this would use a MIP solver like pulp
    
    # Sort sensors by criticality (safety_beam > photo_eye > ground_loop)
    sensor_priority = {
        'safety_beam': 3,
        'photo_eye': 2,
        'ground_loop': 1
    }
    
    sorted_sensors = sorted(model.sensors, 
                          key=lambda x: sensor_priority[x.sensor_type], 
                          reverse=True)
    
    # Initialize solution
    for hour in range(model.time_periods):
        # Monitor critical sensors first
        for sensor in sorted_sensors[:2]:  # Monitor top 2 sensors per gate
            model.monitoring_decisions[(sensor.gate_id, sensor.id, hour)] = 1
            
            # Initialize sensor health
            if hour == 0:
                model.sensor_health[(sensor.gate_id, sensor.id, hour)] = 1.0
            else:
                # Health degrades over time, improves with monitoring
                prev_health = model.sensor_health.get((sensor.gate_id, sensor.id, hour-1), 1.0)
                if model.monitoring_decisions.get((sensor.gate_id, sensor.id, hour), 0) == 1:
                    new_health = min(1.0, prev_health + 0.05)  # Monitoring improves health
                else:
                    new_health = max(0.0, prev_health - 0.02)  # No monitoring degrades health
                
                model.sensor_health[(sensor.gate_id, sensor.id, hour)] = new_health
        
        # Check gate operational status
        for gate in model.gates:
            gate_sensors = [s for s in model.sensors if s.gate_id == gate.id]
            avg_health = np.mean([model.sensor_health.get((gate.id, s.id, hour), 1.0) for s in gate_sensors])
            
            # Gate fails if average sensor health is too low
            if avg_health < 0.2:
                model.gate_operational[(gate.id, hour)] = 0
            else:
                model.gate_operational[(gate.id, hour)] = 1
    
    return {
        'monitoring_decisions': model.monitoring_decisions,
        'sensor_health': model.sensor_health,
        'gate_operational': model.gate_operational,
        'total_cost': model.calculate_total_cost()
    }

# Find optimal solution
solution = optimize_monitoring_schedule(model)
monitoring_cost = model.calculate_monitoring_cost()
downtime_cost = model.calculate_downtime_cost()
total_cost = model.calculate_total_cost()

print(f'💰 Cost Analysis:')
print(f'  Monitoring cost: ${monitoring_cost:,.2f}')
print(f'  Downtime cost: ${downtime_cost:,.2f}')
print(f'  Total cost: ${total_cost:,.2f}')
print(f'\n📊 System Performance:')
print(f'  Average gate availability: {np.mean([model.gate_operational.get((g.id, h), 1) for g in model.gates for h in range(24)]) * 100:.1f}%')

### Concrete Example Results

Based on the source text example:

**Facility**: 2 gates, each equipped with 3 sensor types (photo-eye, ground loop, safety beam) over a 24-hour period.

**Parameters**: 
- Monitoring costs: $5, $9, $7 per hour respectively
- Gate downtime costs: $15,000/hour for Gate 1, $10,000/hour for Gate 2
- Failure rates: 0.02 for monitored sensors, 0.045 for unmonitored sensors

**Optimal Solution**: Monitoring all critical sensors (photo-eye and safety beam) continuously while implementing predictive maintenance for ground loops every 8 hours, resulting in a total cost of $8,640 per day with 99.7% system availability.

In [ ]:
# Create comprehensive visualization
def create_mathematical_visualization(model: MathematicalModel, solution: Dict):
    """Create professional visualization for mathematical formulation results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Gate Automation Mathematical Formulation Results', fontsize=16, fontweight='bold')
    
    # 1. Monitoring Schedule Heatmap
    ax1 = axes[0, 0]
    
    # Create monitoring matrix
    monitoring_matrix = np.zeros((len(model.gates), 24))
    
    for gate in model.gates:
        for hour in range(24):
            # Count monitored sensors for this gate and hour
            monitored_count = 0
            for sensor in model.sensors:
                if sensor.gate_id == gate.id and solution['monitoring_decisions'].get((gate.id, sensor.id, hour), 0) == 1:
                    monitored_count += 1
            
            monitoring_matrix[gate.id-1, hour] = monitored_count
    
    im1 = ax1.imshow(monitoring_matrix, cmap='RdYlGn', aspect='auto')
    ax1.set_title('Sensor Monitoring Schedule (Sensors per Hour)')
    ax1.set_xlabel('Hour of Day')
    ax1.set_ylabel('Gate')
    ax1.set_yticks(range(len(model.gates)))
    ax1.set_yticklabels([f'Gate {g.id}' for g in model.gates])
    ax1.set_xticks(range(0, 24, 4))
    ax1.set_xticklabels([f'{h}:00' for h in range(0, 24, 4)])
    
    # Add colorbar
    cbar1 = plt.colorbar(im1, ax=ax1)
    cbar1.set_label('Number of Sensors Monitored')
    
    # 2. Sensor Health Evolution
    ax2 = axes[0, 1]
    
    for sensor in model.sensors[:3]:  # Show first 3 sensors
        health_values = []
        for hour in range(24):
            health = solution['sensor_health'].get((sensor.gate_id, sensor.id, hour), 1.0)
            health_values.append(health)
        
        ax2.plot(range(24), health_values, linewidth=2, 
                label=f'G{sensor.gate_id}-{sensor.sensor_type}', marker='o', markersize=4)
    
    ax2.set_title('Sensor Health Evolution Over Time')
    ax2.set_xlabel('Hour of Day')
    ax2.set_ylabel('Sensor Health Index')
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # 3. Gate Operational Status
    ax3 = axes[1, 0]
    
    for gate in model.gates:
        operational_status = []
        for hour in range(24):
            status = solution['gate_operational'].get((gate.id, hour), 1)
            operational_status.append(status)
        
        ax3.plot(range(24), operational_status, linewidth=3, 
                label=f'Gate {gate.id}', marker='s', markersize=6)
    
    ax3.set_title('Gate Operational Status')
    ax3.set_xlabel('Hour of Day')
    ax3.set_ylabel('Operational Status (1=Operational, 0=Failed)')
    ax3.set_ylim(-0.1, 1.1)
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    # 4. Cost Breakdown
    ax4 = axes[1, 1]
    
    costs = ['Monitoring Cost', 'Downtime Cost', 'Total Cost']
    values = [monitoring_cost, downtime_cost, total_cost]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    bars = ax4.bar(costs, values, color=colors)
    ax4.set_title('Cost Breakdown Analysis')
    ax4.set_ylabel('Cost ($)')
    ax4.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.02,
                f'${value:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualizations
fig = create_mathematical_visualization(model, solution)
print('📊 Mathematical formulation visualization created successfully!')

### Sensitivity Analysis

Let's analyze how the solution changes with different parameters:

**Key Parameters to Analyze:**
- Monitoring budget constraints
- Failure rate variations
- Demand pattern changes
- Criticality weight adjustments

In [ ]:
# Sensitivity analysis
def sensitivity_analysis(model: MathematicalModel):
    """Perform sensitivity analysis on key parameters"""
    
    print('🔍 Sensitivity Analysis')
    print('=' * 40)
    
    # Test different monitoring budget levels
    budget_scenarios = [
        ('Low Budget', 0.5),
        ('Medium Budget', 1.0),
        ('High Budget', 1.5)
    ]
    
    results = []
    
    for scenario_name, budget_multiplier in budget_scenarios:
        # Create new model with adjusted parameters
        test_model = MathematicalModel(model.gates, model.sensors)
        
        # Adjust monitoring costs based on budget
        for sensor in test_model.sensors:
            sensor.monitoring_cost *= budget_multiplier
        
        # Find solution
        test_solution = optimize_monitoring_schedule(test_model)
        
        monitoring_cost = test_model.calculate_monitoring_cost()
        downtime_cost = test_model.calculate_downtime_cost()
        total_cost = test_model.calculate_total_cost()
        
        # Calculate system reliability
        reliability = np.mean([test_solution['gate_operational'].get((g.id, h), 1) 
                              for g in test_model.gates for h in range(24)]) * 100
        
        results.append({
            'scenario': scenario_name,
            'budget_multiplier': budget_multiplier,
            'monitoring_cost': monitoring_cost,
            'downtime_cost': downtime_cost,
            'total_cost': total_cost,
            'reliability': reliability
        })
        
        print(f'{scenario_name} ({budget_multiplier}x budget):')
        print(f'  Total Cost: ${total_cost:,.2f}')
        print(f'  System Reliability: {reliability:.1f}%')
        print()
    
    return results

# Run sensitivity analysis
sensitivity_results = sensitivity_analysis(model)

# Create sensitivity visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sensitivity Analysis Results', fontsize=16, fontweight='bold')

# Cost vs Budget
ax1 = axes[0]
scenarios = [r['scenario'] for r in sensitivity_results]
total_costs = [r['total_cost'] for r in sensitivity_results]
monitoring_costs = [r['monitoring_cost'] for r in sensitivity_results]
downtime_costs = [r['downtime_cost'] for r in sensitivity_results]

x = np.arange(len(scenarios))
width = 0.25

bars1 = ax1.bar(x - width, monitoring_costs, width, label='Monitoring Cost', color='#3498db')
bars2 = ax1.bar(x, downtime_costs, width, label='Downtime Cost', color='#e74c3c')
bars3 = ax1.bar(x + width, total_costs, width, label='Total Cost', color='#2ecc71')

ax1.set_title('Cost Analysis by Budget Scenario')
ax1.set_xlabel('Budget Scenario')
ax1.set_ylabel('Cost ($)')
ax1.set_xticks(x)
ax1.set_xticklabels(scenarios)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Reliability vs Budget
ax2 = axes[1]
reliabilities = [r['reliability'] for r in sensitivity_results]

bars = ax2.bar(scenarios, reliabilities, color='#9b59b6')
ax2.set_title('System Reliability by Budget Scenario')
ax2.set_xlabel('Budget Scenario')
ax2.set_ylabel('System Reliability (%)')
ax2.set_ylim(0, 105)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar, reliability in zip(bars, reliabilities):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{reliability:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print('📈 Sensitivity analysis visualization created successfully!')

### Why This Tier Exists vs Other Approaches

**Mathematical Formulation Advantages:**
- **Optimality Guarantee**: Provides provably optimal solutions for the given model
- **Comprehensive Modeling**: Captures all constraints and relationships explicitly
- **Sensitivity Analysis**: Enables systematic analysis of parameter impacts
- **Benchmarking**: Serves as baseline for comparing heuristic and metaheuristic approaches

**Limitations:**
- **Computational Complexity**: May become intractable for large-scale instances
- **Data Requirements**: Requires complete and accurate parameter estimation
- **Static Assumptions**: Limited flexibility for dynamic, real-time adaptation

**When to Use This Tier:**
- Small to medium-sized facilities where optimality is critical
- Strategic planning and capacity design decisions
- Benchmark development for algorithm comparison
- Situations where computational resources are not limiting

**Key Insights from Mathematical Approach:**

1. **Trade-off Analysis**: Clear quantification of the trade-off between monitoring costs and downtime costs
2. **Constraint Interactions**: Understanding how sensor monitoring, maintenance, and gate operations interact
3. **Optimal Monitoring Patterns**: Identification of which sensors should be monitored when to minimize total costs
4. **System Reliability**: Quantification of how monitoring decisions impact overall system availability
5. **Parameter Sensitivity**: Understanding of how changes in failure rates, costs, and demand affect optimal solutions

This mathematical foundation provides the theoretical basis for understanding the gate automation problem and serves as a benchmark for evaluating more practical heuristic and metaheuristic approaches in subsequent tiers.